# Results Visualization

This notebook loads the pickle prepared by `evaluate_model.ipynb`.

In [ ]:
import pickle
from pathlib import Path

import ipywidgets as widgets
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

def find_repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'src').is_dir() and (path / 'config.py').is_file():
            return path
    raise RuntimeError('Repository root not found')

REPO_ROOT = find_repo_root()
PICKLE_PATH = REPO_ROOT / 'evaluation_results/run_20260903_170728_best_1000x10_consensus.pkl'
FONT_DIR = REPO_ROOT / 'notebooks/fonts'

font_paths = sorted(path for path in FONT_DIR.iterdir() if path.suffix.lower() == '.ttf')
if len(font_paths) != 4:
    raise RuntimeError(f'Expected four Optima fonts in {FONT_DIR}, found {len(font_paths)}')
for font_path in font_paths:
    font_manager.fontManager.addfont(font_path)

plt.rcParams.update({
    'font.family': 'Optima',
    'mathtext.fontset': 'stix',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#CBD2DC',
    'axes.labelcolor': '#253047',
    'text.color': '#253047',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'axes.unicode_minus': False,
})

with PICKLE_PATH.open('rb') as file:
    payload = pickle.load(file)

if payload.get('schema_version') != 1:
    raise ValueError(f"Unsupported schema version: {payload.get('schema_version')}")

metadata = payload['metadata']
results = payload['results']
plot_data = payload['plot_data']
if not results:
    raise RuntimeError('The pickle does not contain any result')

print(f"Loaded {len(results)} mixtures from {PICKLE_PATH}")
print(f"Run: {metadata['run_dir']}")
print(f"Alignment: {metadata['alignment']}")

## Reconstructions explorer

In [ ]:
CHANNEL_NAMES = ['A real', 'A imag.', 'E real', 'E imag.']
COLORS = {
    'mixture': "#00B342",
    'target': "#2563EB",
    'prediction': '#DC2626',
    'uncertainty': "#E600FF",
}

def style_axis(ax):
    ax.set_facecolor('#FCFCFD')
    ax.grid(color='#CBD5E1', linewidth=0.7, alpha=0.45)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#94A3B8')
    ax.spines['bottom'].set_color('#94A3B8')

def plot_result(result_position, sampling_index):
    result = results[result_position]
    k = result['K']
    sampling_index = min(sampling_index, len(result['aligned_predictions']) - 1)
    mixture = result['mixture']
    target = result['target']
    prediction = result['aligned_predictions'][sampling_index]
    prediction_mean = result['prediction_mean']
    prediction_std = result['prediction_std']
    frequency_bins = np.arange(mixture.shape[-1])
    medoid_index = result['medoid_sampling_index']
    n_rows = k + 2
    n_channels = mixture.shape[0]
    fig, axes = plt.subplots(
        n_rows, n_channels, figsize=(4.8 * n_channels, 2.7 * n_rows),
        sharex=True, squeeze=False,
    )
    for channel in range(n_channels):
        ax = axes[0, channel]
        ax.plot(frequency_bins, mixture[channel], color=COLORS['mixture'], lw=1.7, label='Mixture', zorder=3)
        ax.set_title(CHANNEL_NAMES[channel] if channel < len(CHANNEL_NAMES) else f'Channel {channel + 1}')
        style_axis(ax)
    axes[0, 0].set_ylabel('Mixture')
    for source in range(k):
        row = source + 1
        axes[row, 0].set_ylabel(
            f"Source {source + 1}\nSNR={result['snr'][source]:.2f}\n"
            f"NMSE={result['source_nmse_db'][sampling_index, source]:.2f} dB\n"
            f"std={result['uncertainty_rms'][source]:.3g}\n"
            f"U={100 * result['relative_dispersion_u'][source]:.2f}%"
        )
        for channel in range(n_channels):
            ax = axes[row, channel]
            ax.fill_between(
                frequency_bins,
                prediction_mean[source, channel] - prediction_std[source, channel],
                prediction_mean[source, channel] + prediction_std[source, channel],
                color=COLORS['uncertainty'], alpha=0.18, linewidth=0, label='Mean ±1 std', zorder=1,
            )
            ax.plot(frequency_bins, target[source, channel], color=COLORS['target'], lw=2.2, label='Target', zorder=3)
            ax.plot(frequency_bins, prediction_mean[source, channel], color=COLORS['prediction'], lw=1.4, linestyle=':', label='Sampling mean', zorder=4)
            ax.plot(frequency_bins, prediction[source, channel], color=COLORS['prediction'], lw=1.6, linestyle=(0, (5, 2.5)), label='Selected sampling', zorder=5)
            style_axis(ax)
    residual_row = k + 1
    axes[residual_row, 0].set_ylabel(f"Residual / noise\nMSE={result['residual_mse'][sampling_index]:.3g}")
    for channel in range(n_channels):
        ax = axes[residual_row, channel]
        ax.fill_between(
            frequency_bins,
            prediction_mean[-1, channel] - prediction_std[-1, channel],
            prediction_mean[-1, channel] + prediction_std[-1, channel],
            color=COLORS['uncertainty'], alpha=0.18, linewidth=0, label='Mean ±1 std', zorder=1,
        )
        ax.plot(frequency_bins, target[-1, channel], color=COLORS['target'], lw=2.0, label='Target', zorder=3)
        ax.plot(frequency_bins, prediction_mean[-1, channel], color=COLORS['prediction'], lw=1.4, linestyle=':', label='Sampling mean', zorder=4)
        ax.plot(frequency_bins, prediction[-1, channel], color=COLORS['prediction'], lw=1.6, linestyle=(0, (5, 2.5)), label='Selected sampling', zorder=5)
        ax.set_xlabel('Frequency bin')
        style_axis(ax)
    handles, labels = axes[1, 0].get_legend_handles_labels()
    order = ['Target', 'Selected sampling', 'Sampling mean', 'Mean ±1 std']
    axes[1, 0].legend(
        [handles[labels.index(label)] for label in order], order,
        loc='upper right', frameon=True, facecolor='white', edgecolor='#CBD5E1', framealpha=0.95,
    )
    sampling_label = f'{sampling_index} (medoid)' if sampling_index == medoid_index else str(sampling_index)
    fig.suptitle(
        f"Mixture {result['mixture_index']} — K={k} — sampling {sampling_label} — "
        f"alignment={metadata['alignment']} — loss={result['total_mse'][sampling_index]:.4g}",
        fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()

In [ ]:
element_slider = widgets.IntSlider(
    value=0, min=0, max=len(results) - 1, step=1, description='Element:',
    continuous_update=False, layout=widgets.Layout(width='700px'),
)
sampling_slider = widgets.IntSlider(
    value=results[0]['medoid_sampling_index'], min=0,
    max=len(results[0]['aligned_predictions']) - 1, step=1, description='Sampling:',
    continuous_update=False, layout=widgets.Layout(width='700px'),
)
search_widget = widgets.Text(
    placeholder='Enter an element index', description='Search:',
    continuous_update=False, layout=widgets.Layout(width='700px'),
)
previous_button = widgets.Button(description='◀ Previous')
next_button = widgets.Button(description='Next ▶')
medoid_button = widgets.Button(description='Show medoid')
feedback = widgets.HTML()
font_style = widgets.HTML(value="<style>.jupyter-widgets, .widget-label {font-family: Optima, sans-serif !important;}</style>")

def update_element(change):
    result = results[change['new']]
    sampling_slider.max = len(result['aligned_predictions']) - 1
    sampling_slider.value = result['medoid_sampling_index']
    feedback.value = ''

def search_element(change):
    if not change['new']:
        feedback.value = ''
        return
    try:
        index = int(change['new'])
    except ValueError:
        feedback.value = '<span style="color:#B91C1C">Enter a valid integer</span>'
        return
    if 0 <= index < len(results):
        element_slider.value = index
        feedback.value = ''
    else:
        feedback.value = f'<span style="color:#B91C1C">Index must be between 0 and {len(results) - 1}</span>'

def previous_element(_):
    element_slider.value = max(element_slider.min, element_slider.value - 1)

def next_element(_):
    element_slider.value = min(element_slider.max, element_slider.value + 1)

def show_medoid(_):
    sampling_slider.value = results[element_slider.value]['medoid_sampling_index']

element_slider.observe(update_element, names='value')
search_widget.observe(search_element, names='value')
previous_button.on_click(previous_element)
next_button.on_click(next_element)
medoid_button.on_click(show_medoid)

controls = widgets.VBox([
    font_style,
    widgets.HBox([previous_button, next_button, medoid_button]),
    search_widget, feedback, element_slider, sampling_slider,
])
output = widgets.interactive_output(
    plot_result, {'result_position': element_slider, 'sampling_index': sampling_slider},
)
display(controls, output)

## NMSE and relative dispersion U by K

In [ ]:
REPORT_INK = '#253047'
REPORT_GRID = '#DDE3EA'
CURVE_COLORS = ['#332288', '#88CCEE', '#44AA99', '#117733', '#999933', '#DDCC77', '#CC6677', '#882255', '#AA4499', '#661100']
K_COLORS = dict(zip(range(1, 11), CURVE_COLORS))

def clean_axis(ax):
    ax.grid(axis='y', color=REPORT_GRID, linewidth=0.8)
    ax.grid(axis='x', color='#EEF2F6', linewidth=0.6)
    ax.tick_params(axis='both', length=0)
    ax.spines[['top', 'right']].set_visible(False)

summary = plot_data['by_k']
fig, ax_nmse = plt.subplots(figsize=(9.0, 6.0), dpi=180)
ax_u = ax_nmse.twinx()
ax_nmse.fill_between(summary['K'], summary['nmse_low'], summary['nmse_high'], color='#332288', alpha=0.16, label='NMSE ±1 std')
ax_nmse.plot(summary['K'], summary['nmse_mean'], color='#332288', marker='o', linewidth=2.5, markeredgecolor='white', label='Mean NMSE')
ax_nmse.set_xticks(summary['K'])
ax_nmse.set_xlabel('Number of sources K')
ax_nmse.set_ylabel('NMSE', color='#332288')
ax_nmse.tick_params(axis='y', colors='#332288')
clean_axis(ax_nmse)
ax_u.fill_between(summary['K'], summary['u_low_percent'], summary['u_high_percent'], color='#CC6677', alpha=0.16, label='Medoid-normalized U ±1 std')
ax_u.plot(summary['K'], summary['u_mean_percent'], color='#CC6677', marker='o', linewidth=2.5, markeredgecolor='white', label='Medoid-normalized U')
ax_u.set_ylabel('Medoid-normalized sampling dispersion U [%]', color='#CC6677')
ax_u.tick_params(axis='y', colors='#CC6677', length=0)
ax_u.spines['top'].set_visible(False)
ax_u.spines['right'].set_color('#CC6677')
handles_nmse, labels_nmse = ax_nmse.get_legend_handles_labels()
handles_u, labels_u = ax_u.get_legend_handles_labels()
ax_nmse.legend(handles_nmse + handles_u, labels_nmse + labels_u, loc='upper left', frameon=False, ncol=2)
ax_nmse.set_title('Reconstruction error and sampling dispersion by number of sources', pad=10)
fig.tight_layout()
plt.show()

## NMSE and relative dispersion U by SNR and frequency overlap

In [ ]:
def draw_curves(ax, curves):
    for curve in curves:
        color = K_COLORS[curve['K']]
        ax.plot(curve['curve_x'], curve['curve_y'], color=color, linewidth=2.25, solid_capstyle='round')
        ax.scatter(curve['point_x'], curve['point_y'], color=color, s=25, zorder=3, edgecolor='white', linewidth=0.7)
    clean_axis(ax)

curves = plot_data['curves']
fig, axes = plt.subplots(2, 2, figsize=(16.0, 11.5), dpi=180)
draw_curves(axes[0, 0], curves['nmse_by_snr'])
axes[0, 0].set_title('NMSE by true source SNR')
axes[0, 0].set_ylabel('NMSE')
draw_curves(axes[0, 1], curves['nmse_by_overlap'])
axes[0, 1].set_title('NMSE by pairwise frequency overlap')
axes[0, 1].set_ylabel('NMSE')
draw_curves(axes[1, 0], curves['u_by_snr'])
axes[1, 0].set_title('Sampling dispersion U by true source SNR')
axes[1, 0].set_ylabel('Medoid-normalized U [%]')
draw_curves(axes[1, 1], curves['u_by_overlap'])
axes[1, 1].set_title('Sampling dispersion U by pairwise frequency overlap')
axes[1, 1].set_ylabel('Medoid-normalized U [%]')
for ax in axes[:, 0]:
    ax.set_xlabel('True source SNR')
for ax in axes[:, 1]:
    ax.set_xlabel('Mean pairwise spectral coverage')
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
legend_handles = [
    Line2D([0], [0], color=K_COLORS[int(k)], linewidth=2.25, marker='o', markeredgecolor='white', label=f'K = {int(k)}')
    for k in summary['K']
]
fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=False)
fig.subplots_adjust(bottom=0.12, hspace=0.32, wspace=0.24)
plt.show()